### Importando as bibliotecas básicas

In [ ]:
import pandas as pd
import numpy as np

### 1. Exploração

- Funções como .sample(), .info(), .describe() e .duplicated() ajudam a ter uma visão geral dos dados!

In [ ]:
#Carregando o arquivo
df = pd.read_csv('/content/Google_Ads_Dataset.csv')

In [ ]:
#Recolhendo amostras
df.sample(10)

In [ ]:
#Sumário dos dados
df.info()

In [ ]:
#Estatísticas descritivas
df.describe()

In [ ]:
#Linhas duplicadas
df.duplicated().sum()

### 2. Polimento
- Aqui entram etapas como padronização das colunas e correção de incosistências nos dados;
- **Expressões regulares** (*RegEx*) podem ser muito úteis ⭐ para manipulação de texto.

2.1 RegEx

In [ ]:
#Utilizando RegEx
df[["Cost","Sale_Amount"]] = df[["Cost","Sale_Amount"]].replace("[^0-9.]","", regex = True).astype(float)

2.2 Padronizando colunas com incosistências

In [ ]:
#Colunas Location e Device
print(df["Location"].unique())
print(df["Device"].unique())

In [ ]:
#Correção de incosistências em Location

df["Location"] = df["Location"].str.lower()

ajustar = {
"bengluru": "Bangalore",
"benglore": "Bangalore",
"bangalore": "Bangalore",
"bengaluru": "Bangalore",
"mumbay": "Mumbai",
"mumabi": "Mumbai",
"mumbai": "Mumbai",
"bombay": "Mumbai",
"dheli": "Delhi",
"delhi": "Delhi",
"newdlhi": "New delhi",
"new delhi": "New delhi",
"chnnai": "Chennai",
"madras": "Chennai",
"chenay": "Chennai",
"chennai": "Chennai",
"poona": "Pune",
"punea": "Pune",
"punr": "Pune",
"pune": "Pune"

}

df["Location"] = df["Location"].replace(ajustar)

In [ ]:
#Correção de incosistências em Device
df["Device"] = df["Device"].str.capitalize()

In [ ]:
#Checagem
print(df["Location"].unique())
print(df["Device"].unique())

In [ ]:
#Ajustando o formato de datas
df["Ad_Date"] = pd.to_datetime(df["Ad_Date"], format = "mixed", dayfirst=True)

In [ ]:
#Checagem final
df.head()

### 3. Anomalias
- Na grande maioria dos casos, serão *missing values* e *outliers*;


3.1 Lidando com *Missing Values*:

1°: Verificar se estes dados não podem ser recuperados;

2º: Como estão apresentados no *dataset*? Ex: célula vazia, NaN, ??, #, -...

3º: Identificar qual o tipo de dado faltante;

 4º: Definir o que vai ser feito:
 -  *Drop* de dados;
 - Imputação de estatística simples ➡ média ou mediana (global ou estratificada), imputação por regra...
 - Imputação avançada com *Machine Learning* ➡ imputação múltipla (*MICE*) ou *KNN*

In [ ]:
#Identificando a distribuição de Missing values
(df.isnull().sum()*100/df.index.size).round(2).sort_values(ascending=False) #Em porcentagem

É possível calcular o *Conversion rate* a partir da seguinte fórmula:

$$
CR = \frac{\text{Conversions}}{\text{Clicks}} \times 100
$$

In [ ]:
#Imputação por regra
# 1. Calcula Conversion Rate a partir de Conversions e Clicks
filtro_1 = df["Conversion Rate"].isna() & df["Clicks"].notna() & df["Conversions"].notna()
df.loc[filtro_1, "Conversion Rate"] = df["Conversions"]/df["Clicks"]
# 2. Calcula Conversions a partir de Conversion Rate e Clicks
filtro_2 = df["Conversions"].isna() & df["Clicks"].notna() & df["Conversion Rate"].notna()
df.loc[filtro_2, "Conversions"] = df["Clicks"] * df["Conversion Rate"]
# 3. Calcula Clicks a partir de Conversions e Conversion rate
filtro_3 = df["Clicks"].isna() & df["Conversions"].notna() & df["Conversion Rate"].notna()
df.loc[filtro_3, "Clicks"] = df["Conversions"] / df["Conversion Rate"]


In [ ]:
#Revisando os valores de Missing Values
(df.isnull().sum()*100/df.index.size).round(2).sort_values(ascending=False) #Em porcentagem

In [ ]:
#Imputação estratificada
df_imputado = df.copy()

#Variáveis que possuem faltantes
variaveis = ["Conversion Rate","Conversions", "Clicks", "Sale_Amount", "Impressions", "Leads", "Cost"]

for var in variaveis:
  df_imputado[var] = df_imputado.groupby(["Location","Device"])[var].transform(lambda x: x.fillna(x.median()))

(df_imputado.isnull().sum()*100/df_imputado.index.size).round(2).sort_values(ascending=False)

**3.2 Lidando com Outliers**

1º: Verificar se há outliers:  
- Visualização com gráficos (*Box-plots*, *Scatter plots*)
- Métodos estatísticos (*Z-score* ou *IQR*) ➡ mais formais e precisos ✨
- *Machine Learning* ➡ mais complexos e avançados

In [ ]:
#Identificação de outliers com IQR

outliers = {}

#Filtragem por IQR
for col in df_imputado.select_dtypes(include=["float64", "int64"]).columns:
   Q1 = df_imputado[col].quantile(0.25)
   Q3 = df_imputado[col].quantile(0.75)
   IQR = Q3 - Q1
   limite_inferior = Q1 - 1.5 * IQR
   limite_superior = Q3 + 1.5 * IQR
   filtro = (df_imputado[col] < limite_inferior) | (df_imputado[col] > limite_superior)
   count = filtro.sum()
 #Contagem de Outliers por coluna
   if count > 0:
      menor_valor = df_imputado.loc[filtro, col].min()
      maior_valor = df_imputado.loc[filtro, col].max()
   else:
      maior_valor = None
      menor_valor = None
#Construindo um dataframe com os Outliers encontrados
   outliers[col] = {
       "Contagem de Outliers": count,
       "% de Outliers": ((count / len(df_imputado)) * 100).round(3),
       "Limite Inferior": limite_inferior.round(3),
       "Limite Superior": limite_superior.round(3),
       "Maior outlier": maior_valor,
       "Menor outlier": menor_valor
      }

outlier_df = pd.DataFrame.from_dict(outliers, orient="index")
outlier_df.head()



2º: Julgar se são valores plausíveis ou erros;

3º: Caso sejam erros, definir o que vai ser feito:
- Truncagem ➡ elminar os dados
- Substituição (por média ou mediana) ➡ introduz viés
- Trasformação matemática ➡ reduz a distorção dos *outliers*
- Winsorização ✨
- Algoritmos de *Machine Learning*

In [ ]:
#Passo opcional - Winsorização de Outliers
winsor_df = df_imputado.copy()
for col in winsor_df.select_dtypes(include=["float64", "int64"]).columns:
   Q1 = winsor_df[col].quantile(0.25)
   Q3 = winsor_df[col].quantile(0.75)
   IQR = Q3 - Q1
   limite_inferior = Q1 - 1.5 * IQR
   limite_superior = Q3 + 1.5 * IQR
# Winsorização manual (apenas se houver outliers)
   winsor_df[col] = winsor_df[col].apply(lambda x: limite_inferior if x < limite_inferior else x and\
   limite_superior if x > limite_superior else x)

In [ ]:
#Checagem
winsor_df.describe()

### 4. Finalização

In [ ]:
winsor_df.to_csv('Google_Ads_Limpo.csv', index=False)